<a href="https://colab.research.google.com/github/namii07/Namisha-Codeboosters-Internship-2026/blob/main/Phase_02_GenAI/Day_08_RAG_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here is the exact text transcribed from the "Key Vocabulary" table shown in the image:

---

## Key Vocabulary

| Term | Meaning |
| --- | --- |
| **Chunk** | A small piece of a document (e.g. one paragraph) |
| **Embedding** | A vector (list of numbers) representing text meaning |
| **Vector Database** | A database that stores and searches vectors by similarity |
| **Retrieval** | Finding the most relevant chunks for a given query |
| **Context Injection** | Adding retrieved chunks into the LLM's prompt |
| **Grounding** | Forcing the LLM to answer based on provided facts |
| **Knowledge Base** | The collection of documents the system can retrieve from |

---



In [4]:
!pip install sentence-transformers chromadb groq pandas -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 108.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curren

In [5]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print("All Libraries imported succesfully , Ready to build a RAG system")

All Libraries imported succesfully , Ready to build a RAG system


API KEY : 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

In [6]:
GROQ_API_KEY = "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"
os.environ["GROQ_API_KEY"] =GROQ_API_KEY

groq_client = Groq(api_key=GROQ_API_KEY)
print("Groq API client initialized")


Groq API client initialized


PART 4: Loading the Knowledge Base

Our knowledge base is the college_notes.csv file from Day 7.

It contains 15 notes across subjects: Data Engineering, Machine Learning, GenAI, and Python.

Each row in the CSV represents one document (one knowledge chunk).

What is a Knowledge Base?

Simple English: A library of documents that the AI can look through when answering questions.

Analogy: Like a textbook that the AI can open and read before answering your question.

Technical: A collection of text documents indexed in a vector database for similarity search.

In [7]:
df = pd.read_csv('college_notes.csv')

print("Shape of dataset:", df.shape)

print("\nColumn names:" , df.columns.tolist())

print("first 3 rows:")
print(df.head(3))




Shape of dataset: (15, 4)

Column names: ['note_id', 'subject', 'topic', 'content']
first 3 rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [8]:
print("Subjects in the dataset:")
print(df['subject'].value_counts())

print("\n Sample of topics:")
print(df[['note_id' , 'subject' , 'topic']].to_string(index=False))

print("\nLength of content (number of characters) for each note:")

df['content_length'] = df['content'].apply(len)
print(df[['topic', 'content_length']].to_string(index=False))


Subjects in the dataset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

 Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Pytho

CHUNKING

In [9]:
documents = df['content'].tolist()

ids = [f"note_{row['note_id']}" for row in df.to_dict('records')]

metadatas = [
  { "subject": row['subject'] , "topic": row['topic']}
  for row in df.to_dict('records')
]

print(f"Total chunks prepared: {len(documents)}")
print(f"First document ID : {ids[0]}")
print(f"First metadata: {metadatas[0]}")
print(f"First 100 chars of doc : { documents[0][:100]}...")


Total chunks prepared: 15
First document ID : note_N001
First metadata: {'subject': 'Data Engineering', 'topic': 'ETL Pipelines'}
First 100 chars of doc : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sourc...


In [10]:
print("Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("\nEmbedding model loaded successfully")
test_embedding = embedding_model.encode("this is a test sentence")
print(f"Test embedding shape: {test_embedding.shape}")
print(f"First 5 values of test ebedding: {test_embedding[:5]}")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


Embedding model loaded successfully
Test embedding shape: (384,)
First 5 values of test ebedding: [0.07155243 0.06848023 0.00660337 0.10176966 0.01112225]


In [11]:
chroma_client = chromadb.Client()

collection = chroma_client.get_or_create_collection(name = "college_notes_rag")
print("ChromaDB client created")
print(f"Collection name: {collection.name}")
print(f"Number of documents in collection: {collection.count()}")


ChromaDB client created
Collection name: college_notes_rag
Number of documents in collection: 0


In [12]:
print("Generating embeddings for all 15 notes...")
print("This may take 15-30 seconds...")

embeddings = embedding_model.encode(documents, show_progress_bar=True)
print(f"\n Embedding matrix shape : {embeddings.shape}")

embeddings_list = embeddings.tolist()

collection.add(
    documents=documents,
    embeddings=embeddings_list,
    metadatas=metadatas,
    ids=ids
)

print(f"\n Documents succesfully added to ChromaDB")
print(f"Total documents in collection: {collection.count()}")


Generating embeddings for all 15 notes...
This may take 15-30 seconds...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


 Embedding matrix shape : (15, 384)

 Documents succesfully added to ChromaDB
Total documents in collection: 15


In [13]:
def retrieve_relevant_chunks(question , top_k=3):
  question_embedding = embedding_model.encode(question).tolist()

  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k
 )

  return results
print("Retrieval Function defined succesfully")
print("Function: retrieve_relevant_chunks(question , top_5=3)")

Retrieval Function defined succesfully
Function: retrieve_relevant_chunks(question , top_5=3)


In [14]:
test_question = "what is ETL and how does it work in data engineering?"
print(f"Test Question: {test_question}")
print("=" * 60)

results =  retrieve_relevant_chunks(test_question , top_k=3)

print("\n Top 3 Retrieved Chunks")
print("=" * 60)

for i ,(doc, dist, meta) in enumerate(zip(results['documents'][0] , results['distances'][0], results['metadatas'][0])):

   print(f"\nResult {i+1}:")
   print(f" Subject : {meta['subject']}")
   print(f"Topic   : {meta['topic']}")
   print(f" Distance : {dist:.4f}")

   print(f" Content : {doc[:120]}...")



Test Question: what is ETL and how does it work in data engineering?

 Top 3 Retrieved Chunks

Result 1:
 Subject : Data Engineering
Topic   : ETL Pipelines
 Distance : 0.2269
 Content : ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i...

Result 2:
 Subject : Data Engineering
Topic   : APIs and Data Collection
 Distance : 1.0690
 Content : An API or Application Programming Interface allows two software applications to talk to each other. In data engineering ...

Result 3:
 Subject : Python Programming
Topic   : Data Visualization
 Distance : 1.3375
 Content : Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo...


CONTEXT INJECTION : we take the retrieved documents and paste them into the LLM's Prompt



Here is the exact text transcribed from the "The RAG Prompt Template" section shown in the image:

---

## The RAG Prompt Template

**SYSTEM:**
You are a helpful academic assistant. Answer questions based ONLY on the provided context. If the answer is not in the context, say "I don't have enough information to answer this." Do not use your general training knowledge. Only use the context provided.

**USER:**
Context:
---


Retrieved Document 1


---


Retrieved Document 2


---


Retrieved Document 3


---

Question: [User's Question]

Answer:

In [15]:
def build_context_from_results(results):

  context_parts = []

  for i , (doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):

      chunk_text = f"[Source {i+1}:{meta['subject']} - {meta['topic']}]\n{doc}"
      context_parts.append(chunk_text)
  context_str = "\n\n---\n\n".join(context_parts)
  return context_str
context = build_context_from_results(results)
print("built context string from retrieved chunks:")
print("=" * 60)
print(context[:500] + "...")
print(f"\nTotal context length: {len(context)} characters")



built context string from retrieved chunks:
[Source 1:Data Engineering - ETL Pipelines]
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.

---

[Source 2:Data Engineering - APIs and Data Collection]
An API or Application Programming Interface allows two software applications to talk to each other. In data engineering APIs are used to fetch data from external services like we...

Total context length: 853 characters


In [16]:
def generate_rag_answer(question , context):

  system_prompt = """ You are a helpful academic assistant for engineering students.

  You will be guven context retrieved from a college knowledge base, and a student's question.

  RULES:
  1. Answer ONLY using the information provided in the context below.
  2. If the answer is not found in the context, say exactly:
     "I don't have enough information in my knowledge base to answer this question."
  3. Do not use your general training knowledge.
  4. Keep answers clear, accurate , and beginner-friendly,
  5. Mention which source the information came from when possible."""

  user_prompt = f"""Context from knowledge base:

{context}

---

Student's Question: {question}

Please answer the question based only on the context provided above. """
  response = groq_client.chat.completions.create(
    model = "llama-3.1-8b-instant",
    messages=[
        {"role":"system" , "content":system_prompt},
        {"role":"user" , "content":user_prompt}
      ],
      temperature = 0.1,
       max_tokens = 500,
  )

  answer = response.choices[0].message.content
  return answer

print("RAG generation function defined.")

RAG generation function defined.


In [19]:
def ask_college_assistant(question, top_k=3 , verbose=True):
  if verbose:
    print(f"Question: {question}")
    print("=" * 60)
    print("Step 1: Retrieving relevant documents...")

  results = retrieve_relevant_chunks(question, top_k=top_k)

  if verbose:
    print(f"Retrieved {top_k} chunks from the knowledge base:")
    for i, meta in enumerate(results['metadatas'][0]):
      print(f" {i+1}. {meta['subject']}- {meta['topic']}")
    print("\n Step 2: Building context string...")

  context = build_context_from_results(results)

  if verbose:
    print(f"Context built ({len(context)} characters)")
    print("\n Step 3: Sending to LLM for answer generation...")

  answer = generate_rag_answer(question,context)

  if verbose:
    print("\n" + "=" *60)
    print("ANSWER:")
    print("=" * 60)
    print(answer)
    print("=" * 60)

  return answer

print("Complete RAG pipeline function ready")
print("Function: ask_college_assistant(question , top_k=3 , verbose=True)")

Complete RAG pipeline function ready
Function: ask_college_assistant(question , top_k=3 , verbose=True)


In [20]:
question_1 ="What is ETL and what are its three main stages?"

answer_1 = ask_college_assistant(question_1, top_k=3, verbose=True)

Question: What is ETL and what are its three main stages?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
 1. Data Engineering- ETL Pipelines
 2. Generative AI- Retrieval Augmented Generation
 3. Generative AI- Prompt Engineering

 Step 2: Building context string...
Context built (927 characters)

 Step 3: Sending to LLM for answer generation...

ANSWER:
According to the context from [Source 1: Data Engineering - ETL Pipelines], ETL stands for Extract Transform Load. 

The three main stages of ETL are:

1. Extract: This stage involves collecting raw data from different sources.
2. Transform: This stage involves transforming the raw data into a clean and structured format.
3. Load: This stage involves loading the transformed data into a database or data warehouse for analysis.

Source: [Source 1: Data Engineering - ETL Pipelines]


In [21]:
question_2 = "How do embeddings help in building search systems?"

answer_2 = ask_college_assistant(question_2, top_k=3, verbose=True)


Question: How do embeddings help in building search systems?
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
 1. Generative AI- Retrieval Augmented Generation
 2. Generative AI- Large Language Models
 3. Machine Learning- Feature Engineering

 Step 2: Building context string...
Context built (907 characters)

 Step 3: Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.


In [22]:
question_3 = "What is the Population of Tokyo"

print("Testing with an out-of-scope question(not in college notes):")
answer_3 = ask_college_assistant(question_3, top_k=3, verbose=True)

Testing with an out-of-scope question(not in college notes):
Question: What is the Population of Tokyo
Step 1: Retrieving relevant documents...
Retrieved 3 chunks from the knowledge base:
 1. Generative AI- Large Language Models
 2. Data Engineering- SQL Databases
 3. Data Engineering- Data Cleaning

 Step 2: Building context string...
Context built (796 characters)

 Step 3: Sending to LLM for answer generation...

ANSWER:
I don't have enough information in my knowledge base to answer this question.

The context provided does not contain any information about the population of Tokyo or any other geographical location. It only discusses Large Language Models, SQL databases, and data cleaning.


## Comparison Table

| Feature | Without RAG | With RAG |
| --- | --- | --- |
| Knowledge source | LLM training data (fixed) | Your custom documents (updatable) |
| Hallucination risk | High | Low |
| Can use private data | No | Yes |
| Knowledge cutoff | Yes (training date) | No (you add new docs anytime) |
| Cites sources | No | Yes (you know which chunk was used) |
| Cost | Cheaper (shorter prompts) | Slightly higher (longer prompts with context) |

### Real-World Data Engineering RAG Applications

RAG is not just for chatbots. It is widely used in Data Engineering:

---

#### 1. Data Catalog Assistants

Large companies have thousands of datasets. Engineers ask:
*"What does the customer_churn table contain?"*
RAG retrieves from the data catalog documents and answers accurately.

---

#### 2. Pipeline Debugging Assistants

When a data pipeline fails, engineers ask:
*"Why did the ETL job fail with error code 504?"*
RAG retrieves from past incident reports, runbooks, and error logs.

---

#### 3. SQL Generation from Documentation

Engineers ask: *"Write a query to get revenue by region from the sales schema"*
RAG retrieves schema documentation and the LLM generates accurate SQL.

---

### 4. Data Governance Q&A

Compliance teams ask: "What is our data retention policy for Pll data?"
RAG retrieves from governance policy documents

In [23]:
def retrieve_by_subject(question , subject_filter , top_k=2):

  question_embedding = embedding_model.encode(question).tolist()

  results = collection.query(
      query_embeddings = [question_embedding],
      n_results = top_k,
      where = {"subject" : subject_filter}
  )

  return results

print("Retrieving only from GenAI subject")
print("=" * 50)

filtered_results = retrieve_by_subject(
    question = "How do LLMs generate text?",
    subject_filter = "GenAI",
    top_k = 2
)

for i, (doc, meta) in enumerate(zip(
    filtered_results['documents'][0],
    filtered_results['metadatas'][0]
)):

    print(f"Result {i+1}: [{meta['subject']}] {meta['topic']}")
    print(f"  {doc[:100]}...")







Retrieving only from GenAI subject


### Beginner Questions

**Q1.** What is hallucination in the context of LLMs?

**Q2.** What does RAG stand for? What problem does it solve?

**Q3.** What is the role of a vector database in the RAG pipeline?

---

### Intermediate Questions

**Q4.** What is the difference between the Indexing phase and the Querying phase of RAG?

**Q5.** Why must you use the same embedding model for both documents and queries?

**Q6.** Why is a low temperature (e.g. 0.1) preferred for RAG-based LLM calls?

---

### Coding Questions

**Q7.** Modify the ask_college_assistant() function to also display the distance scores of retrieved chunks in the output

**Q8**  Change the system prompt in generate_rag_answer() to instruct the LLM to always respond in bullet points

**Q9** Add a function that returns only the topic names of retrieved chunks without their full content

**Q1. What is hallucination in the context of LLMs?**

Answer:
Hallucination occurs when a Large Language Model (LLM) generates information that sounds correct but is actually false, misleading, or not supported by facts.

Example:
If an LLM says a college offers a course that does not exist, it is hallucinating.

**Q2. What does RAG stand for? What problem does it solve?**

Answer:
RAG stands for Retrieval-Augmented Generation.

It solves the problem of LLMs lacking access to specific or up-to-date information by retrieving relevant documents from an external knowledge base and providing them as context before generating an answer.

Benefits:

Reduces hallucinations
Provides more accurate answers
Uses domain-specific knowledge
Keeps information up-to-date without retraining the model


**Q3. What is the role of a vector database in the RAG pipeline?**

Answer:
A vector database stores document embeddings (vector representations of text) and enables efficient similarity search.

Role in RAG:

Store embeddings of documents/chunks.
Convert user query into an embedding.
Find the most similar document chunks.
Return relevant chunks to the LLM as context.

Examples:

ChromaDB
Pinecone
Weaviate
FAISS
Intermediate Questions


**Q4. What is the difference between the Indexing phase and the Querying phase of RAG?**

Indexing Phase	Querying Phase
Happens before users ask questions	Happens when a user asks a question
Documents are collected and processed	User query is processed
Documents are split into chunks	Query is converted to embedding
Embeddings are generated for chunks	Similar chunks are retrieved
Embeddings are stored in vector DB	Retrieved chunks are sent to LLM
Flow

Indexing

Documents
   ↓
Chunking
   ↓
Embeddings
   ↓
Vector Database

Querying

User Query
   ↓
Embedding
   ↓
Similarity Search
   ↓
Relevant Chunks
   ↓
LLM Answer


**Q5. Why must you use the same embedding model for both documents and queries?**

Answer:
The same embedding model must be used so that documents and queries are represented in the same vector space.

If different models are used:

Vector dimensions may differ.
Similarity calculations become unreliable.
Retrieval quality decreases significantly.

Example:
If documents are embedded using all-MiniLM-L6-v2, queries should also use all-MiniLM-L6-v2.

**Q6. Why is a low temperature (e.g., 0.1) preferred for RAG-based LLM calls?**

Answer:
A low temperature makes the model more deterministic and focused on the provided context.

Advantages:
Reduces hallucinations
Produces consistent answers
Sticks closely to retrieved documents
Improves factual accuracy
Example:
temperature = 0.1

This is preferred because RAG systems prioritize correctness over creativity.

In [24]:
def ask_college_assistant():
    question = input("Ask a question: ")

    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    print("\nRetrieved Chunks:\n")

    for i, doc in enumerate(results["documents"][0]):
        distance = results["distances"][0][i]

        print(f"Chunk {i+1}")
        print(f"Distance Score: {distance}")
        print(doc)
        print("-" * 50)

    context = "\n".join(results["documents"][0])

    answer = generate_rag_answer(question, context)

    print("\nAnswer:")
    print(answer)

In [25]:
system_prompt = """
You are a helpful academic assistant for engineering students.

You will be given context retrieved from a college knowledge base and a student's question.

RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context, say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Always respond using bullet points.
4. Do not make up information.
"""

In [26]:
def get_retrieved_topics(question):
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    topics = []

    for metadata in results["metadatas"][0]:
        topics.append(metadata["topic"])

    return topics

### MINI PROJECT:

### Project Description

Build a complete **College Knowledge Assistant** that:

1. Loads the `college_notes.csv` knowledge base
2. Indexes all notes in ChromaDB with embeddings
3. Accepts a student question
4. Retrieves the top 3 relevant notes
5. Injects them as context into a Groq LLM prompt
6. Returns a clear, grounded answer with source citations
7. Handles questions outside the knowledge base gracefully

In [43]:
import pandas as pd

# This cell successfully loads the 'college_notes.csv' knowledge base,
# completing the first step of the mini-project.
df = pd.read_csv("college_notes.csv")
print(df.head())

  note_id           subject                     topic  \
0    N001  Data Engineering             ETL Pipelines   
1    N002  Data Engineering             SQL Databases   
2    N003  Data Engineering             Data Cleaning   
3    N004  Data Engineering  APIs and Data Collection   
4    N005  Data Engineering      Big Data and PySpark   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  
3  An API or Application Programming Interface al...  
4  Big Data refers to extremely large datasets th...  


In [28]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("college_notes")

for i, row in df.iterrows():
    collection.add(
        ids=[str(i)],
        documents=[row["content"]],
        metadatas=[{
            "topic": row["topic"],
            "subject": row["subject"]
        }]
    )

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 43.8MiB/s]


In [30]:
def retrieve_notes(question):
    results = collection.query(
        query_texts=[question],
        n_results=3
    )

    return results

In [40]:
from groq import Groq

client = Groq(api_key="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")

def generate_rag_answer(question, context):

    system_prompt = """
    You are a helpful academic assistant.

    Answer ONLY using the provided context.

    If the answer is not present in the context,
    say:
    'I don't have enough information in my knowledge base to answer this question.'
    """

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",
             "content": f"Context:\n{context}\n\nQuestion:\n{question}"}
        ],
        temperature=0.1
    )

    return response.choices[0].message.content

In [36]:
def get_sources(results):
    return [
        metadata["topic"]
        for metadata in results["metadatas"][0]
    ]

In [37]:
def ask_college_assistant():

    question = input("Ask a question: ")

    results = retrieve_notes(question)

    context = "\n".join(results["documents"][0])

    answer = generate_rag_answer(question, context)

    sources = get_sources(results)

    print("\nAnswer:")
    print(answer)

    print("\nSources:")
    for source in sources:
        print("-", source)

In [42]:
ask_college_assistant()

Ask a question: What is ETL?

Answer:
ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources, transforming it into a clean and structured format, and loading it into a database or data warehouse for analysis.

Sources:
- ETL Pipelines
- APIs and Data Collection
- Retrieval Augmented Generation
